In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [19]:
class BatsmanState(TypedDict):
  balls: int
  sixes: int
  fours: int
  runs: int

  sr: float
  bpb: float
  boundry_percent: float
  summary: str

In [21]:
def calculate_sr(state: BatsmanState):
  sr = (state['runs']/state['balls'])*100
  return {'sr': sr}

In [22]:
def calculate_bpb(state: BatsmanState):
  bpb = state['balls']/(state['fours'] + state['sixes'])
  return {'bpb': bpb}

In [23]:
def calculate_boundry_percent(state: BatsmanState):
  boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6))/state['runs'])*100
  return {'boundry_percent': boundary_percent}

In [24]:
def gen_summary(state: BatsmanState):
  summary = f"""
Strike Rate: {state['sr']} \n
Balls Per Boundry: {state['bpb']} \n
Boundry Percent: {state['boundry_percent']} \n
"""
  return {'summary': summary}

In [25]:
graph = StateGraph(BatsmanState)
#add nodes
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundry_percent', calculate_boundry_percent)
graph.add_node('gen_summary', gen_summary)
#add edges
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundry_percent')

graph.add_edge('calculate_sr', 'gen_summary')
graph.add_edge('calculate_bpb', 'gen_summary')
graph.add_edge('calculate_boundry_percent', 'gen_summary')

graph.add_edge('gen_summary', END)
workflow = graph.compile()

In [26]:
initial_state = {
  'balls': 100,
  'sixes': 7,
  'fours': 5,
  'runs': 120
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'balls': 100, 'sixes': 7, 'fours': 5, 'runs': 120, 'sr': 120.0, 'bpb': 8.333333333333334, 'boundry_percent': 51.66666666666667, 'summary': '\nStrike Rate: 120.0 \n\nBalls Per Boundry: 8.333333333333334 \n\nBoundry Percent: 51.66666666666667 \n\n'}
